# 01. 날씨 x 택시 수요 분석 (ASOS 기반)

## 분석 배경 및 목적

택시는 날씨에 민감한 대표적 수요응답형 교통수단이다. 비, 눈, 폭염, 한파 등 기상 이벤트가 발생하면 보행과 대중교통 접근성이 저하되어 택시 수요가 급변하지만, 그 크기와 방향은 시간대, 요일, 계절에 따라 상이하다. 이 분석은 다음 세 가지 질문에 답한다.

1. **날씨 조건별 수요 차이**: 맑음/비/눈/폭염/한파 등 기상 상태에 따라 택시 수요가 얼마나 달라지는가?
2. **강수 수요반응 계수**: 시간대 고정효과를 통제했을 때 강수량 1mm 증가가 택시 수요에 미치는 순효과(탄력성)는 얼마인가?
3. **시계열 추세와 날씨 교란**: 코로나, 계절성 등 시기 효과를 흡수한 뒤에도 날씨의 순효과가 유의한가?

이러한 분석은 날씨 조건부 수요 예측 모델의 기초 자료가 되며, 택시 배차 최적화와 운전자 수급 계획에 직접 활용할 수 있다.

**방법론적 근거**: Liu et al. (2025)은 기상 조건을 그래프 네트워크의 조건 변수로 활용하여 수요 예측 정확도를 개선하였으며, Nasser et al. (2025)은 교통-기상 데이터 융합이 예측 성능을 유의하게 높임을 실증하였다. 본 분석은 이들 연구의 접근을 따라 ASOS 관측 자료를 시간 단위로 결합하되, 시간대 고정효과 회귀를 통해 출퇴근 시간대의 교란(수요와 강수가 동시 상승하는 문제)을 통제한다.

- **내부 데이터**: DC_TBYXD012 (요금정보) -- 승차건수, 시간대, 결제금액, 운행거리
- **외부 데이터**:
  - `weather_asos_hourly_seoul_2018_2026.csv` -- 시간별 기온/강수/습도/풍속
  - `weather_asos_daily_seoul_2018_2026.csv` -- 일별 적설(snow_depth), 강수
  - `calendar_2018_2026.csv` -- 요일/주말/공휴일
- **분석 목표**: 날씨 조건별 수요 차이, 강수 수요반응 계수(시간대 고정효과 통제), 시계열 추이

> 컬럼명은 모두 DATA_DICTIONARY.md / 테이블정의서 기준.
> 시간별 ASOS에는 눈(적설)이 없으므로 일별 `snow_depth`를 date로 조인한다.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import platform
import warnings
warnings.filterwarnings('ignore')

# 한글 폰트
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
else:
    plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 100

In [ ]:
# === 메모리 유틸 ===
import gc, psutil, os

def mem_usage(tag=''):
    gb = psutil.Process(os.getpid()).memory_info().rss / 1024**3
    print(f'[MEM {tag}] {gb:.2f} GB')

CHUNK_SIZE = 1_000_000
mem_usage('start')

## 1. D012 시간대별 집계 (청크 가중집계)

6억 건 이상의 요금정보를 메모리 제약 없이 처리하기 위해 청크 단위 가중집계를 수행한다. 시간대(hour)별 집계는 이후 날씨 조건 조인 및 고정효과 회귀의 기본 단위가 된다.

In [ ]:
# === 경로 설정 (폐쇄망 환경에 맞게 수정) ===
D012_PATH      = './DC_TBYXD012.csv'
EXT_DIR        = './external_data'
WEATHER_HOURLY = f'{EXT_DIR}/weather_asos_hourly_seoul_2018_2026.csv'
WEATHER_DAILY  = f'{EXT_DIR}/weather_asos_daily_seoul_2018_2026.csv'
CALENDAR       = f'{EXT_DIR}/calendar_2018_2026.csv'

# PAY_AMT=결제금액(단건), RIDE_DIST=승차거리. RIDE_DTIME=YYYYMMDDHHMMSS(14)
usecols = ['RIDE_DTIME', 'ALIGHT_DTIME', 'PAY_AMT', 'RIDE_DIST']
dtypes  = {'RIDE_DTIME': str, 'ALIGHT_DTIME': str, 'PAY_AMT': 'float64', 'RIDE_DIST': 'float64'}

# 청크별 sum/count를 따로 누적 → 마지막에 한 번만 나눠야 가중평균이 정확
parts = []
for chunk in pd.read_csv(D012_PATH, usecols=usecols, dtype=dtypes, chunksize=CHUNK_SIZE):
    rd = pd.to_datetime(chunk['RIDE_DTIME'], format='%Y%m%d%H%M%S', errors='coerce')
    m = rd.notna()
    chunk, rd = chunk[m].copy(), rd[m]
    chunk['date'] = rd.dt.normalize()
    chunk['hour'] = rd.dt.hour

    ad = pd.to_datetime(chunk['ALIGHT_DTIME'], format='%Y%m%d%H%M%S', errors='coerce')
    dur = (ad - rd).dt.total_seconds() / 60
    chunk['dur'] = dur.where((dur >= 0) & (dur < 600))  # 음수/이상치 제거(0~10시간)

    g = chunk.groupby(['date', 'hour']).agg(
        ride_count=('PAY_AMT', 'size'),
        fare_sum=('PAY_AMT', 'sum'),
        dist_sum=('RIDE_DIST', 'sum'),
        dur_sum=('dur', 'sum'),
        dur_cnt=('dur', 'count'),
    ).reset_index()
    parts.append(g)
    del chunk, rd, ad, dur, g
    gc.collect()

agg = pd.concat(parts, ignore_index=True).groupby(['date', 'hour']).sum().reset_index()
agg['avg_fare']     = agg['fare_sum'] / agg['ride_count']
agg['avg_dist']     = agg['dist_sum'] / agg['ride_count']
agg['avg_duration'] = agg['dur_sum']  / agg['dur_cnt']
hourly_demand = agg.drop(columns=['fare_sum', 'dist_sum', 'dur_sum', 'dur_cnt'])
del parts, agg
gc.collect()

print(f"D012 시간대별 집계: {len(hourly_demand):,}행, "
      f"{hourly_demand['date'].min().date()} ~ {hourly_demand['date'].max().date()}")
mem_usage('after load')
hourly_demand.head()

## 2. 외부 데이터 조인 (ASOS 시간별/일별 + 캘린더)

기상청 ASOS 자동기상관측 자료를 시간 단위로 조인한다. ASOS는 전국 95개 지점에서 1시간 간격으로 기온, 강수량, 상대습도, 풍속 등을 관측하며, 본 분석에서는 서울(108) 지점 데이터를 사용한다. 일별 적설량(snow_depth)은 시간별 데이터에 포함되지 않으므로 별도 일별 파일에서 date 기준으로 조인한다. 캘린더(공휴일/요일) 정보를 함께 결합하여 이후 교차 분석의 기반을 마련한다.

In [ ]:
# 시간별 ASOS (명세서 컬럼: date, hour, temp, rainfall, humidity, wind_speed ...)
wh = pd.read_csv(WEATHER_HOURLY, parse_dates=['date'])
wh['hour'] = wh['hour'].astype(int)                       # '00'~'23' 문자열 → int
wh = wh.rename(columns={'temp': 'temperature', 'rainfall': 'precipitation'})
wh['precipitation'] = wh['precipitation'].fillna(0)       # 강수 빈값 = 0

# 일별 ASOS에서 적설만 (시간별엔 눈이 없음)
wd = pd.read_csv(WEATHER_DAILY, parse_dates=['date'])[['date', 'snow_depth']]
wd['snow_depth'] = wd['snow_depth'].fillna(0)

# 캘린더 (요일/주말/공휴일)
cal = pd.read_csv(CALENDAR, parse_dates=['date'])[['date', 'day_name', 'is_weekend', 'is_holiday']]

merged = (hourly_demand
    .merge(wh[['date', 'hour', 'temperature', 'precipitation', 'humidity', 'wind_speed']],
           on=['date', 'hour'], how='inner')
    .merge(wd, on='date', how='left')
    .merge(cal, on='date', how='left'))
merged['snow_depth'] = merged['snow_depth'].fillna(0)

print(f"조인 결과: {len(merged):,}행 (시간별 매칭 inner join)")
print(f"기온 결측: {merged['temperature'].isna().sum()}, 습도 결측: {merged['humidity'].isna().sum()}")
merged.head()

## 3. 날씨 조건 분류

연속형 기상 변수를 범주형 날씨 조건으로 변환한다. 분류 기준은 기상청 특보 기준 및 선행연구(Liu et al., 2025)의 threshold를 참고하여 설정하였다. 범주화를 통해 각 날씨 상태의 수요 차이를 직관적으로 비교할 수 있으며, 이후 시간대 고정효과 모형에서 범주형 더미변수로 활용한다.

In [ ]:
def classify_weather(r):
    if r['snow_depth'] > 0:            return '눈'
    if r['precipitation'] >= 10:       return '강한비(10mm+)'
    if r['precipitation'] >= 1:        return '보통비(1~10mm)'
    if r['precipitation'] > 0:         return '약한비(<1mm)'
    return '맑음'

merged['weather_condition'] = merged.apply(classify_weather, axis=1)

# 기온 구간 (명세서 temp_category 기준에 맞춤)
merged['temp_bin'] = pd.cut(merged['temperature'],
    bins=[-30, -10, 0, 10, 25, 30, 45],
    labels=['혹한(<-10)', '추위(-10~0)', '서늘(0~10)', '쾌적(10~25)', '더움(25~30)', '폭염(>30)'])

merged['day_type'] = merged['is_weekend'].map({0: '평일', 1: '주말'})

print(merged['weather_condition'].value_counts(), '\n')
print(merged['temp_bin'].value_counts().sort_index())

## 4. 날씨 조건별 시간대별 수요

날씨 조건(맑음/비/눈/폭염/한파 등)과 시간대의 교차표를 통해 각 기상 상태가 시간대별 수요 프로파일을 어떻게 변형하는지 시각화한다. 이 분석은 Nasser et al. (2025)이 제시한 "기상 조건이 시간대별 교통 패턴의 shape를 변경한다"는 가설을 서울 택시 데이터에서 검증하는 것에 해당한다. 실무적으로는 날씨 예보에 따른 시간대별 배차 수량 조정의 근거가 된다.

In [ ]:
WC_ORDER  = ['맑음', '약한비(<1mm)', '보통비(1~10mm)', '강한비(10mm+)', '눈']
WC_COLORS = {'맑음':'#2196F3','약한비(<1mm)':'#90CAF9','보통비(1~10mm)':'#FF9800',
             '강한비(10mm+)':'#F44336','눈':'#9C27B0'}

wh_hour = merged.groupby(['weather_condition', 'hour'])['ride_count'].mean().reset_index()

fig, ax = plt.subplots(figsize=(14, 7))
for c in WC_ORDER:
    s = wh_hour[wh_hour['weather_condition'] == c]
    if len(s):
        ax.plot(s['hour'], s['ride_count'], marker='o', label=c, color=WC_COLORS[c], lw=2)
ax.set_xlabel('시간대'); ax.set_ylabel('평균 승차건수')
ax.set_title('날씨 조건별 시간대별 택시 수요', fontweight='bold')
ax.set_xticks(range(24)); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 5. 강수 수요반응 -- 시간대 고정효과 통제

**방법론**: 단순 회귀(수요 ~ 강수량)는 출퇴근 시간대에 수요와 강수가 동시에 높아지는 허위상관(spurious correlation)을 발생시킨다. 이를 해결하기 위해 시간대(hour) 고정효과를 포함한 패널 회귀 모형을 적용한다.

$$demand_{d,h} = \alpha_h + \beta \cdot precip_{d,h} + \epsilon_{d,h}$$

여기서 $\alpha_h$는 시간대 고정효과, $\beta$는 강수의 순효과(수요반응 계수)이다. 이 접근은 Liu et al. (2025)이 multi-graph 네트워크에서 시간대를 조건 변수로 분리한 것과 동일한 논리이며, 계량경제학의 고정효과(Fixed Effects) 추정과 일치한다.

기존 분석은 '비 오는 시간만' 뽑아 단순회귀로 탄력성을 구해, 출퇴근(수요와 강수 동시 상승) 효과가 교란되었다. 여기서는 **같은 시간대(hour) 안에서 맑음 대비 비/눈의 수요 차이**를 구해 시간대 효과를 통제한다.

In [ ]:
# 시간대별 맑음 기준선
base = (merged[merged['weather_condition'] == '맑음']
        .groupby('hour')['ride_count'].mean().rename('clear'))

rows = []
for c in WC_ORDER[1:]:
    sub = merged[merged['weather_condition'] == c].groupby('hour')['ride_count'].mean()
    j = pd.concat([base, sub.rename('cond')], axis=1).dropna()
    if len(j):
        chg = (j['cond'] / j['clear'] - 1) * 100
        rows.append({'날씨조건': c,
                     '시간대통제 평균 변화율(%)': round(chg.mean(), 1),
                     '최대증가 시간대': f'{int(chg.idxmax())}시',
                     '최대 변화율(%)': round(chg.max(), 1),
                     '관측 시간수': int(merged[merged.weather_condition==c].shape[0])})
resp = pd.DataFrame(rows)
print('=== 강수/강설 수요반응 (시간대 통제, 맑음 대비) ===')
resp

In [ ]:
# 강수량 구간별 (전체 평균 + 시간대 통제 평균 병행)
merged['precip_bin'] = pd.cut(merged['precipitation'],
    bins=[-0.1, 0, 1, 5, 10, 20, 1000],
    labels=['0mm','0~1mm','1~5mm','5~10mm','10~20mm','20mm+'])

# 시간대 통제: 각 시간대 내 0mm 평균으로 정규화 후 구간 평균
hour_base = merged[merged['precipitation'] == 0].groupby('hour')['ride_count'].mean()
merged['ride_norm'] = merged['ride_count'] / merged['hour'].map(hour_base)

pb = merged.groupby('precip_bin', observed=True).agg(
    avg_rides=('ride_count', 'mean'),
    norm_ratio=('ride_norm', 'mean'),
    n=('ride_count', 'count')).reset_index()
pb['시간대통제 변화율(%)'] = ((pb['norm_ratio'] - 1) * 100).round(1)

fig, ax = plt.subplots(figsize=(11, 6))
colors = ['#2196F3' if v >= 0 else '#F44336' for v in pb['시간대통제 변화율(%)']]
ax.bar(pb['precip_bin'].astype(str), pb['시간대통제 변화율(%)'], color=colors)
ax.axhline(0, color='black', lw=0.5)
ax.set_xlabel('강수량 구간'); ax.set_ylabel('수요 변화율 (%)')
ax.set_title('강수량 구간별 수요 변화율 (시간대 통제, 0mm 대비)', fontweight='bold')
for i, v in enumerate(pb['시간대통제 변화율(%)']):
    ax.text(i, v + (0.5 if v>=0 else -1.5), f'{v:+.1f}%', ha='center', fontweight='bold')
ax.grid(alpha=0.3, axis='y')
plt.tight_layout(); plt.show()
pb[['precip_bin', 'avg_rides', '시간대통제 변화율(%)', 'n']]

## 6. 기온 구간별 시간대별 수요

기온을 구간화(binning)하여 한파, 쾌적, 폭염 등 온도 범주별 수요 프로파일을 분석한다. 극한 기온(영하 10도 이하, 영상 33도 이상)은 보행 환경을 악화시켜 택시 전환 수요를 유발하며, 이 효과는 시간대에 따라 비대칭적으로 나타날 수 있다.

In [ ]:
TEMP_ORDER = ['혹한(<-10)','추위(-10~0)','서늘(0~10)','쾌적(10~25)','더움(25~30)','폭염(>30)']
TEMP_COLORS = dict(zip(TEMP_ORDER,
    ['#1A237E','#5C6BC0','#42A5F5','#66BB6A','#FF9800','#F44336']))

th = merged.groupby(['temp_bin','hour'], observed=True)['ride_count'].mean().reset_index()
fig, ax = plt.subplots(figsize=(14, 7))
for t in TEMP_ORDER:
    s = th[th['temp_bin'] == t]
    if len(s):
        ax.plot(s['hour'], s['ride_count'], marker='o', label=t, color=TEMP_COLORS[t], lw=2)
ax.set_xlabel('시간대'); ax.set_ylabel('평균 승차건수')
ax.set_title('기온 구간별 시간대별 택시 수요', fontweight='bold')
ax.set_xticks(range(24)); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 7. 평일 vs 주말 x 날씨 교차 분석

평일과 주말의 택시 수요 구조가 다르므로, 날씨 효과도 요일 유형에 따라 상이할 수 있다. 평일은 통근 수요가 지배적이어서 날씨에 의한 모드 전환(대중교통 -> 택시)이 크고, 주말은 여가 수요가 중심이어서 악천후 시 외출 자체가 줄어 수요가 감소할 수 있다. 이 교차 분석은 날씨 x 요일 상호작용 효과를 확인한다.

In [ ]:
cross = (merged.groupby(['day_type','weather_condition'])['ride_count'].mean()
         .unstack(fill_value=0).reindex(columns=WC_ORDER))
cross_pct = cross.div(cross['맑음'], axis=0).subtract(1).multiply(100).round(1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
cross.plot(kind='bar', ax=ax1, rot=0, color=[WC_COLORS[c] for c in WC_ORDER])
ax1.set_title('평일/주말 × 날씨별 평균 승차건수', fontweight='bold')
ax1.set_ylabel('평균 승차건수'); ax1.legend(title='날씨'); ax1.grid(alpha=0.3, axis='y')

cross_pct.drop(columns='맑음').plot(kind='bar', ax=ax2, rot=0)
ax2.set_title('평일/주말 × 날씨별 수요 변화율 (맑음 대비)', fontweight='bold')
ax2.set_ylabel('변화율 (%)'); ax2.axhline(0, color='black', lw=0.5)
ax2.legend(title='날씨'); ax2.grid(alpha=0.3, axis='y')
plt.tight_layout(); plt.show()
cross_pct

## 8. 시계열 추이 — 일별 + 7일 이동평균 + 강수 매칭 비교

추이를 핵심 산출물로 보강한다.
1. 일별 총수요에 7·30일 이동평균을 얹어 장기 추세를 본다.
2. **같은 달·같은 요일 안에서** 비 온 날 vs 안 온 날 수요를 짝지어 비교(matched)하여,
   코로나·계절 등 시기 효과를 흡수한 '날씨 순효과'를 본다.

In [ ]:
# 일별 총수요
daily = merged.groupby('date').agg(
    rides=('ride_count', 'sum'),
    precip=('precipitation', 'sum'),
    day_name=('day_name', 'first'),
    is_weekend=('is_weekend', 'first')).reset_index()
daily = daily.set_index('date').asfreq('D')
daily['rides'] = daily['rides'].interpolate()
daily['ma7']  = daily['rides'].rolling(7, center=True).mean()
daily['ma30'] = daily['rides'].rolling(30, center=True).mean()

fig, ax = plt.subplots(figsize=(18, 6))
ax.plot(daily.index, daily['rides'], lw=0.3, alpha=0.4, color='gray', label='일별')
ax.plot(daily.index, daily['ma7'],  lw=1.2, color='steelblue', label='7일 이동평균')
ax.plot(daily.index, daily['ma30'], lw=2,   color='darkorange', label='30일 이동평균')
ax.set_title('택시 일별 수요 추이', fontweight='bold')
ax.set_xlabel('날짜'); ax.set_ylabel('일 승차건수'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# 매칭 비교: 같은 (연-월, 요일) 셀 내에서 비온 날 vs 안온 날 평균
daily['ym'] = daily.index.to_period('M').astype(str)
daily['rainy_day'] = daily['precip'] > 0

cells_ = (daily.dropna(subset=['rides'])
          .groupby(['ym', 'day_name', 'rainy_day'])['rides'].mean().unstack('rainy_day'))
cells_ = cells_.dropna()                      # 두 조건 모두 존재하는 셀만
cells_.columns = ['no_rain', 'rain']
cells_['diff_pct'] = (cells_['rain'] / cells_['no_rain'] - 1) * 100

print(f'매칭된 (월×요일) 셀 수: {len(cells_)}')
print(f'비 온 날 수요 순효과(중앙값): {cells_["diff_pct"].median():+.1f}%')
print(f'비 온 날 수요 순효과(평균):   {cells_["diff_pct"].mean():+.1f}%')

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(cells_['diff_pct'], bins=30, color='steelblue', edgecolor='black', lw=0.5)
ax.axvline(cells_['diff_pct'].median(), color='red', ls='--',
           label=f'중앙값 {cells_["diff_pct"].median():+.1f}%')
ax.axvline(0, color='black', lw=0.8)
ax.set_xlabel('비 온 날 수요 변화율 (%) — 같은 월·요일 내 매칭')
ax.set_ylabel('셀 수'); ax.set_title('강수의 수요 순효과 분포 (시기 효과 통제)', fontweight='bold')
ax.legend(); ax.grid(alpha=0.3, axis='y')
plt.tight_layout(); plt.show()

## 9. 요약

본 분석의 핵심 발견과 실무적 시사점을 정리한다.

**실무 활용**: 날씨-수요 반응 계수는 기상청 단기 예보와 결합하여 향후 3-6시간 택시 수요를 예측하는 데 활용할 수 있다. 특히 강수 시 시간대별 수요 증가 패턴은 플랫폼 배차 알고리즘의 서지 프라이싱(surge pricing) 기준 설계에 직접 반영 가능하다.

In [ ]:
summary = merged.groupby('weather_condition').agg(
    평균_승차건수=('ride_count', 'mean'),
    평균_결제금액=('avg_fare', 'mean'),
    평균_운행거리=('avg_dist', 'mean'),
    관측_시간수=('ride_count', 'count')).round(1).reindex(WC_ORDER)
b = summary.loc['맑음', '평균_승차건수']
summary['수요_변화율(%)'] = ((summary['평균_승차건수'] / b - 1) * 100).round(1)

print('=' * 60)
print('날씨 × 택시 수요 분석 요약')
print('=' * 60)
print('· 강수 순효과(매칭, 위 셀): 시기·요일 통제 후 추정치')
print('· avg_fare는 PAY_AMT(결제금액) 평균 — 운임 분해는 05/11 노트북 참조')
summary

In [ ]:
del merged, daily
gc.collect()
mem_usage('final')

## 11. [보강] 시계열 추세

In [ ]:
# === [시계열 보강] 일별 수요 추세 (7·30일 이동평균) ===
# 기존 분석과 독립적으로 일별 시계열을 다시 집계해 장기 추세를 확인한다.
import pandas as _pd, numpy as _np, matplotlib.pyplot as _plt
_daily = {}
for _ck in _pd.read_csv(D012_PATH if 'D012_PATH' in dir() else './DC_TBYXD012.csv',
                        usecols=['RIDE_DTIME'], dtype={'RIDE_DTIME': str}, chunksize=1_000_000):
    _d = _ck['RIDE_DTIME'].str[:8]
    _d = _d[_d.str.match(r'\d{8}')]
    for _k, _v in _d.groupby(_d).size().items():
        _daily[_k] = _daily.get(_k, 0) + _v
    del _ck
_ts = _pd.Series(_daily); _ts.index = _pd.to_datetime(_ts.index, format='%Y%m%d')
_ts = _ts.sort_index().asfreq('D').interpolate()
_ma7, _ma30 = _ts.rolling(7, center=True).mean(), _ts.rolling(30, center=True).mean()
fig, ax = _plt.subplots(figsize=(18, 5))
ax.plot(_ts.index, _ts.values, lw=0.3, alpha=0.4, color='gray', label='일별')
ax.plot(_ma7.index, _ma7.values, lw=1.2, color='steelblue', label='7일 이동평균')
ax.plot(_ma30.index, _ma30.values, lw=2, color='darkorange', label='30일 이동평균')
ax.set_title('일별 택시 수요 추세 (7·30일 이동평균)', fontweight='bold')
ax.set_xlabel('날짜'); ax.set_ylabel('일 건수'); ax.legend(); ax.grid(alpha=0.3)
_plt.tight_layout(); _plt.show()
print(f"기간 {_ts.index.min().date()} ~ {_ts.index.max().date()}, 일평균 {_ts.mean():,.0f}건")

---

## References

1. Liu, Y., Zhang, K., Li, Y., Yan, Z., & Gao, J. (2025). Weather-Conditioned Multi-graph Network for Ride-Hailing Demand Forecasting. *Proceedings of the International Conference on Service-Oriented Computing (ICSOC 2024)*, Springer.
2. Nasser, R., Saunier, N., & Morency, C. (2025). Traffic and Weather Data Fusion for Traffic Prediction: A Deep Learning Approach. *Wiley Interdisciplinary Reviews: Data Mining and Knowledge Discovery*.
3. 기상청 (2024). 종관기상관측(ASOS) 자료 이용 안내. https://data.kma.go.kr
4. Angrist, J. D., & Pischke, J.-S. (2009). *Mostly Harmless Econometrics*. Princeton University Press. (고정효과 추정 방법론)